[author]
Computer Vision Tracking Lab Metrics Report

In [2]:
!pip -q install ultralytics psutil pandas tqdm


In [ ]:
import time, os, math, platform
import pandas as pd
import psutil
from tqdm import tqdm

from ultralytics import YOLO

#Optional NVIDIA GPU metrics - I am using CPU AMD RYZEN 7 chip
try:
    import pynvml
    pynvml.nvmlInit()
    _HAS_NVML = True
except Exception:
    _HAS_NVML = False

def get_cpu_metrics():
    # CPU utilization over a short interval (non-blocking-ish)
    cpu_pct = psutil.cpu_percent(interval=None)
    freq = psutil.cpu_freq()
    freq_mhz = freq.current if freq else None
    ram = psutil.virtual_memory()
    return {
        "cpu_pct": cpu_pct,
        "cpu_freq_mhz": freq_mhz,
        "ram_used_gb": ram.used / (1024**3),
        "ram_pct": ram.percent
    }

def get_gpu_metrics():
    if not _HAS_NVML:
        return {"gpu_util_pct": None, "vram_used_gb": None, "vram_total_gb": None}
    h = pynvml.nvmlDeviceGetHandleByIndex(0)
    util = pynvml.nvmlDeviceGetUtilizationRates(h)
    mem = pynvml.nvmlDeviceGetMemoryInfo(h)
    return {
        "gpu_util_pct": float(util.gpu),
        "vram_used_gb": mem.used / (1024**3),
        "vram_total_gb": mem.total / (1024**3),
    }


In [4]:
def run_benchmark(
    model_path: str,
    source: str,
    task: str = "track",     # "track" or "predict"
    imgsz: int = 640,
    conf: float = 0.25,
    iou: float = 0.5,
    max_frames: int = 300,   # benchmark first N frames for consistency
    warmup_frames: int = 20,
):
    model = YOLO(model_path)

    # choose method
    if task == "track":
        iterator = model.track(
            source=source,
            stream=True,
            persist=True,
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            verbose=False
        )
    else:
        iterator = model.predict(
            source=source,
            stream=True,
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            verbose=False
        )

    frame_times = []
    cpu_samples, gpu_samples = [], []

    t0 = time.perf_counter()
    for idx, r in enumerate(iterator):
        t1 = time.perf_counter()
        # time per yielded frame result
        frame_times.append(t1 - t0)
        t0 = t1

        # sample system metrics (every ~10 frames to reduce overhead)
        if idx % 10 == 0:
            cpu_samples.append(get_cpu_metrics())
            gpu_samples.append(get_gpu_metrics())

        if idx >= max_frames - 1:
            break

    # drop warmup
    usable = frame_times[warmup_frames:] if len(frame_times) > warmup_frames else frame_times
    if len(usable) == 0:
        raise RuntimeError("No frames processed (check your video path / webcam permissions).")

    avg_dt = sum(usable) / len(usable)
    fps = 1.0 / avg_dt

    # summarize system metrics
    def avg_key(samples, key):
        vals = [s[key] for s in samples if s.get(key) is not None]
        return sum(vals)/len(vals) if vals else None

    summary = {
        "model": model_path,
        "task": task,
        "imgsz": imgsz,
        "conf": conf,
        "iou": iou,
        "frames": len(usable),
        "fps": fps,
        "avg_frame_ms": avg_dt * 1000.0,

        "cpu_pct_avg": avg_key(cpu_samples, "cpu_pct"),
        "cpu_freq_mhz_avg": avg_key(cpu_samples, "cpu_freq_mhz"),
        "ram_used_gb_avg": avg_key(cpu_samples, "ram_used_gb"),
        "ram_pct_avg": avg_key(cpu_samples, "ram_pct"),

        "gpu_util_pct_avg": avg_key(gpu_samples, "gpu_util_pct"),
        "vram_used_gb_avg": avg_key(gpu_samples, "vram_used_gb"),
        "vram_total_gb": avg_key(gpu_samples, "vram_total_gb"),
    }
    return summary


In [ ]:
source = "video.mp4"   # or 0 for webcam

models = [
    "yolov8n.pt",
    "yolov8s.pt",
    "yolov8m.pt",
    "yolov8l.pt",
    # "yolov8x.pt",
]

rows = []
for m in models:
    print("Running:", m)
    rows.append(run_benchmark(
        model_path=m,
        source=source,
        task="track",     # or "predict"
        imgsz=640,
        max_frames=300
    ))

df = pd.DataFrame(rows).sort_values("fps", ascending=False)
df


In [ ]:
out_path = "yolo_benchmark_results.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

df[["model", "fps", "avg_frame_ms", "cpu_pct_avg", "gpu_util_pct_avg", "vram_used_gb_avg"]]

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(df["model"], df["fps"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("FPS")
plt.title("YOLO Tracking FPS by Model Size")
plt.tight_layout()
plt.show()